In [1]:
import sys
sys.path.append("..")  # needed to import dataloader.py

import torch
import torchvision
from dataloader import MultiTaskBrainDataset
from torch.utils.data import DataLoader
from torchinfo import summary
torch.manual_seed(42)  # set seed for reproducibility

device = torch.device("cpu" if not torch.cuda.is_available() else "cuda") 

In [2]:
class ConvBlock(torch.nn.Module):
    """Two conv layers with GN + ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            torch.nn.GroupNorm(4, out_ch),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            torch.nn.GroupNorm(4, out_ch),
            torch.nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class TinyUNet(torch.nn.Module):
    """
    Lightweight U-Net for CPU inference.

    Encoder: 3 levels
    Skip connections preserved for segmentation quality.

    Input:  (B, 1, H, W)
    Output: (B, num_classes, H, W) — raw logits
    """
    def __init__(self, num_classes=1, base_ch=12):
        super().__init__()

        
        self.enc1 = ConvBlock(1,         base_ch)      
        self.enc2 = ConvBlock(base_ch,   base_ch*2)    
        self.enc3 = ConvBlock(base_ch*2, base_ch*4)   
        self.pool = torch.nn.MaxPool2d(2)

        
        self.bottleneck = ConvBlock(base_ch*4, base_ch*8)

        
        self.up3  = torch.nn.ConvTranspose2d(base_ch*8, base_ch*4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(base_ch*8, base_ch*4)   

        self.up2  = torch.nn.ConvTranspose2d(base_ch*4, base_ch*2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base_ch*4, base_ch*2)   

        self.up1  = torch.nn.ConvTranspose2d(base_ch*2, base_ch,   kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base_ch*2, base_ch)  

        
        self.head = torch.nn.Conv2d(base_ch, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))

        x = self.bottleneck(self.pool(s3))

        x = self.dec3(torch.cat([self.up3(x), s3], dim=1))
        x = self.dec2(torch.cat([self.up2(x), s2], dim=1))
        x = self.dec1(torch.cat([self.up1(x), s1], dim=1))

        return self.head(x)
    
class DiceBCELoss(torch.nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, eps=1e-6, pos_weight=10.0):
        super().__init__()
        self.bce_weight  = bce_weight
        self.dice_weight = dice_weight
        self.eps         = eps
        self.pos_weight  = pos_weight

    def forward(self, pred_logits, target):
        probs = torch.sigmoid(pred_logits)

        # Weighted BCE
        bce = -(target * torch.log(probs + 1e-6) * self.pos_weight +
                (1 - target) * torch.log(1 - probs + 1e-6))
        bce_loss = bce.mean()

        # Dice
        intersection = (probs * target).sum(dim=(1,2,3))
        union        = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
        dice_loss    = 1 - (2*intersection + self.eps) / (union + self.eps)
        dice_loss    = dice_loss.mean()

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

In [3]:
model = TinyUNet(num_classes=1, base_ch=12)
model.to(device)
print(summary(model, input_size=(4, 1, 256, 256)))

Layer (type:depth-idx)                   Output Shape              Param #
TinyUNet                                 [4, 1, 256, 256]          --
├─ConvBlock: 1-1                         [4, 12, 256, 256]         --
│    └─Sequential: 2-1                   [4, 12, 256, 256]         --
│    │    └─Conv2d: 3-1                  [4, 12, 256, 256]         108
│    │    └─GroupNorm: 3-2               [4, 12, 256, 256]         24
│    │    └─ReLU: 3-3                    [4, 12, 256, 256]         --
│    │    └─Conv2d: 3-4                  [4, 12, 256, 256]         1,296
│    │    └─GroupNorm: 3-5               [4, 12, 256, 256]         24
│    │    └─ReLU: 3-6                    [4, 12, 256, 256]         --
├─MaxPool2d: 1-2                         [4, 12, 128, 128]         --
├─ConvBlock: 1-3                         [4, 24, 128, 128]         --
│    └─Sequential: 2-2                   [4, 24, 128, 128]         --
│    │    └─Conv2d: 3-7                  [4, 24, 128, 128]         2,592
│    │  

In [4]:
from torch.utils.data import DataLoader
import os
import json

train_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="segmentation",
    train_or_test="train",
    subset="train",
    augment=True,
    seed=42,
    new_height=256, 
    new_width=256
)

val_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="segmentation",
    train_or_test="train",
    subset="val",
    augment=False,
    seed=42,
    new_height=256, 
    new_width=256
)

train_loader = DataLoader(train_data, batch_size=4, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_data,   batch_size=4, shuffle=False, num_workers=0)

loss_fn   = DiceBCELoss(bce_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.Adam(model.parameters())

def dice_coefficient(pred_logits, target, threshold=0.5, eps=1e-6):
    """
    Computes the Dice score between predicted binary mask and ground truth.
    pred_logits: raw model output (B, 1, H, W)
    target:      ground truth mask (B, 1, H, W), values in {0, 1}
    """
    probs = torch.sigmoid(pred_logits)
    preds = (probs > threshold).float()
    intersection = (preds * target).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (union + eps)
    return dice.mean().item()

best_val_dice  = 0.0
checkpoint_dir = os.path.join("..", "results", "segmentation", "tumor_segmentation_v2_grant")
os.makedirs(checkpoint_dir, exist_ok=True)

best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
metadata_path   = os.path.join(checkpoint_dir, "training_metadata.json")

training_metadata = {
    "best_val_dice": 0.0,
    "best_epoch":    None,
    "history":       []
}

for epoch in range(100):
    model.train()
    running_loss = 0.0
    running_dice = 0.0

    for inputs, masks in train_loader:
        inputs, masks = inputs.to(device), masks.float().to(device)

        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_dice += dice_coefficient(predictions, masks)

    epoch_loss = running_loss / len(train_loader)
    epoch_dice = running_dice / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for inputs, masks in val_loader:
            inputs, masks = inputs.to(device), masks.float().to(device)
            predictions = model(inputs)
            loss = loss_fn(predictions, masks.float())
            val_loss += loss.item()
            val_dice += dice_coefficient(predictions, masks)

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {epoch_loss:.4f} | Train Dice: {epoch_dice:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}"
    )

    # Record epoch metrics in metadata
    training_metadata["history"].append({
        "epoch":      epoch + 1,
        "train_loss": round(epoch_loss, 6),
        "train_dice": round(epoch_dice, 6),
        "val_loss":   round(val_loss,   6),
        "val_dice":   round(val_dice,   6),
    })

    # Save best model and update metadata if improved
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        training_metadata["best_val_dice"] = round(val_dice, 6)
        training_metadata["best_epoch"]    = epoch + 1

        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_dice":             val_dice,
            "val_loss":             val_loss,
        }, best_model_path)
        print(f"  -> Best model saved (epoch {epoch+1}, val_dice={val_dice:.4f})")

    # Write metadata JSON after every epoch
    with open(metadata_path, "w") as f:
        json.dump(training_metadata, f, indent=4)

Epoch 001 | Train Loss: 0.5712 | Train Dice: 0.2476 | Val Loss: 0.4600 | Val Dice: 0.4009
  -> Best model saved (epoch 1, val_dice=0.4009)
Epoch 002 | Train Loss: 0.4245 | Train Dice: 0.4337 | Val Loss: 0.3600 | Val Dice: 0.4967
  -> Best model saved (epoch 2, val_dice=0.4967)
Epoch 003 | Train Loss: 0.3643 | Train Dice: 0.5004 | Val Loss: 0.3252 | Val Dice: 0.5350
  -> Best model saved (epoch 3, val_dice=0.5350)
Epoch 004 | Train Loss: 0.3315 | Train Dice: 0.5299 | Val Loss: 0.2949 | Val Dice: 0.5650
  -> Best model saved (epoch 4, val_dice=0.5650)
Epoch 005 | Train Loss: 0.3089 | Train Dice: 0.5533 | Val Loss: 0.2771 | Val Dice: 0.5968
  -> Best model saved (epoch 5, val_dice=0.5968)
Epoch 006 | Train Loss: 0.2951 | Train Dice: 0.5675 | Val Loss: 0.2740 | Val Dice: 0.5790
Epoch 007 | Train Loss: 0.2868 | Train Dice: 0.5792 | Val Loss: 0.2564 | Val Dice: 0.6032
  -> Best model saved (epoch 7, val_dice=0.6032)
Epoch 008 | Train Loss: 0.2780 | Train Dice: 0.5898 | Val Loss: 0.2581 | Val

In [5]:
test_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="segmentation",
    train_or_test="test",
    augment=False,
    seed=42,
    new_height=256,
    new_width=256
)

test_loader = DataLoader(test_data, batch_size=4, shuffle=False, num_workers=0)

# Load best checkpoint
checkpoint_dir  = os.path.join("..", "results", "segmentation", "tumor_segmentation_v2_grant")
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

checkpoint = torch.load(best_model_path, map_location=device)
model = TinyUNet(num_classes=1, base_ch=12)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

print(f"Loaded best model from epoch {checkpoint['epoch']} "
      f"(val_dice={checkpoint['val_dice']:.4f})")

model.eval()

test_loss = 0.0
test_dice = 0.0
all_preds = []
all_masks = []

with torch.no_grad():
    for inputs, masks in test_loader:
        inputs, masks = inputs.to(device), masks.to(device)
        predictions = model(inputs)
        loss = loss_fn(predictions, masks.float())

        test_loss += loss.item()
        test_dice += dice_coefficient(predictions, masks)

        binary_preds = (torch.sigmoid(predictions) > 0.5).float().cpu()
        all_preds.append(binary_preds)
        all_masks.append(masks.cpu())

test_loss /= len(test_loader)
test_dice /= len(test_loader)

print(f"\nTest Loss: {test_loss:.4f} | Test Dice: {test_dice:.4f}")

from sklearn.metrics import precision_score, recall_score, f1_score

all_preds_flat = torch.cat(all_preds).view(-1).numpy().astype(int)
all_masks_flat = torch.cat(all_masks).view(-1).numpy().astype(int)

precision = precision_score(all_masks_flat, all_preds_flat, zero_division=0)
recall    = recall_score   (all_masks_flat, all_preds_flat, zero_division=0)
f1        = f1_score       (all_masks_flat, all_preds_flat, zero_division=0)

print(f"Pixel Precision : {precision:.4f}")
print(f"Pixel Recall    : {recall:.4f}")
print(f"Pixel F1 Score  : {f1:.4f}")

Loaded best model from epoch 89 (val_dice=0.8389)


C:\Users\glawl\AppData\Local\Temp\ipykernel_26776\2975565186.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=devic


Test Loss: 0.1786 | Test Dice: 0.8439
Pixel Precision : 0.8549
Pixel Recall    : 0.8887
Pixel F1 Score  : 0.8715
